In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


<h1>For dummy submission initially</h1>

In [6]:
import pandas as pd

data = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")
data.to_csv("submission.csv", index = False)

<h1>Importing all dependencies required:</h1>

In [9]:
import pandas as pd
import numpy as np
import string
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

<h1>The following is for Milestone 1</h1>
<hr>
<h3>Question 1:</h3> Calculate the frequency distribution of the correct  answer  (A, B, C, D, E) in train.csv. Based on your counts, what is the sum of the occurrences of the most frequent option and the least frequent option?  

In [11]:
counts = df['answer'].value_counts()
sum_most_and_least = counts.max() + counts.min()
sum_most_and_least

814

<h3>Question 2:</h3> After converting the prompt column to lowercase and removing all standard punctuation characters (using Python's string.punctuation), split the text by whitespace. What is the total number of unique words (vocabulary size) across the entire cleaned prompt column of train.csv?  

In [12]:
def clean_text(text):
    text = text.lower()
    return text.translate(str.maketrans('', '', string.punctuation))

cleaned_prompts = df['prompt'].apply(clean_text)
vocab = set(' '.join(cleaned_prompts).split())
len(vocab)

859

<h3>Question 3:</h3> Using the cleaned prompt from Row ID 1, filter out the standard English stop words using sklearn.feature_extraction.text.ENGLISH_STOP_WORDS. How many words are left in the prompt for Row ID 1 after filtering?  

In [13]:
row1_tokens = clean_text(df.iloc[0]['prompt']).split()
filtered_tokens = [w for w in row1_tokens if w not in ENGLISH_STOP_WORDS]
len(filtered_tokens)

13

<h3>Question 4:</h3> Fit a default TfidfVectorizer(stop_words='english') on a list containing all the combined text of the prompts and options in train.csv. What is the exact total number of feature columns (vocabulary size) generated by the vectorizer?  

In [14]:
combined_text = df['prompt'] + " " + df['A'] + " " + df['B'] + " " + df['C'] + " " + df['D'] + " " + df['E']
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(combined_text)
len(vectorizer.get_feature_names_out())

2762

<h3>Question 5:</h3> Using the TF-IDF vectorizer fitted in Question 3, calculate the cosine similarity between the prompt and option A strictly for Row ID 1. What is the resulting similarity score? (Round to 4 decimal places).  

In [15]:
prompt_vec = vectorizer.transform([df.iloc[0]['prompt']])
option_a_vec = vectorizer.transform([df.iloc[0]['A']])
round(cosine_similarity(prompt_vec, option_a_vec)[0][0], 4)

np.float64(0.272)

<h3>Question 6:</h3> Expand the logic from Question 4: For every row in train.csv, calculate the cosine similarity between the prompt and each of its 5 options .  Then calculate the percentage of instances where the option with the highest cosine similarity matches the correct answer.   

In [16]:
def get_best_option(row):
    options = ['A', 'B', 'C', 'D', 'E']
    prompt_v = vectorizer.transform([row['prompt']])
    scores = {opt: cosine_similarity(prompt_v, vectorizer.transform([row[opt]]))[0][0] for opt in options}
    return max(scores, key=scores.get)

matches = df.apply(lambda row: get_best_option(row) == row['answer'], axis=1)
matches.mean() * 100

np.float64(13.55)

<h3>Question 7:</h3> If the ground truth answer for a question is C, what is the MAP@3 score if a model predicts C A B?  

In [17]:
gt_7 = 'C'
pred_7 = ['C', 'A', 'B']

score_7 = 1 / (pred_7.index(gt_7) + 1) if gt_7 in pred_7[:3] else 0.0

score_7

1.0

<h3>Question 8:</h3> The Majority Class Baseline: Find the most frequent correct answer in the training set (using your data from Q1). Make a static prediction for every single row where that most frequent answer is your 1st guess, followed by the second most frequent, and then the third most frequent. What is the overall MAP@3 score of this "Majority Class" baseline on train.csv?

In [18]:
gt_8 = 'B'
pred_8 = ['D', 'B', 'E']

score_8 = 1 / (pred_8.index(gt_8) + 1) if gt_8 in pred_8[:3] else 0.0
score_8

0.5

<h3>Question 9:</h3> The Majority Class Baseline: Find the most frequent correct answer in the training set (using your data from Q1). Make a static prediction for every single row where that most frequent answer is your 1st guess, followed by the second most frequent, and then the third most frequent. What is the overall MAP@3 score of this "Majority Class" baseline on train.csv?

In [22]:
top_3_answers = df['answer'].value_counts().index[:3].tolist()
baseline_map3 = df['answer'].apply(
    lambda ans: 1.0 if ans == top_3_answers[0] else 
               (0.5 if ans == top_3_answers[1] else 
               (1/3 if ans == top_3_answers[2] else 0.0))
).mean()
baseline_map3

np.float64(0.42125)

<h3>Question 10:</h3> The TF-IDF Pipeline: Build a basic pipeline that evaluates every row in train.csv. For each row, calculate the TF-IDF cosine similarity between the prompt and each of the 5 options. Sort these options from highest similarity to lowest to form your top 3 predictions. What is the final average MAP@3 score of this TF-IDF pipeline across the entire training set?  

In [18]:
def get_top3_preds(row):
    options = ['A', 'B', 'C', 'D', 'E']
    p_v = vectorizer.transform([row['prompt']])
    s = {opt: cosine_similarity(p_v, vectorizer.transform([row[opt]]))[0][0] for opt in options}
    return sorted(s, key=s.get, reverse=True)

all_scores = df.apply(lambda row: map_at_3(row['answer'], get_top3_preds(row)), axis=1)
all_scores.mean()

np.float64(0.2961666666666667)